In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder, SystemMessagePromptTemplate, HumanMessagePromptTemplate
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
generator_llm = init_chat_model("google_genai:gemini-2.5-flash")
evaluator_llm = init_chat_model("google_genai:gemini-2.5-flash")
optimizer_llm = init_chat_model("google_genai:gemini-2.5-flash")

In [12]:
from pydantic import BaseModel, Field
from typing import Literal


class tweetevaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"]
    feedback: str


In [13]:
structured_evaluator_llm = evaluator_llm.with_structured_output(tweetevaluation)

In [4]:
def Tweetstate(TypedDict):
    topic: str
    tweet : str
    evaluation : Literal["approved", "needs_improvement"] 
    feedback : str
    iteration : int
    max_iterations : int

In [5]:
def Generate_tweet(state:Tweetstate):
    prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template("You are a helpful assistant that generates tweets based on a given topic."),
        HumanMessagePromptTemplate.from_template("Generate a tweet about the following topic: {topic}"),
    ])
    response = generator_llm.invoke(prompt).content
    return {"tweet": response}
    

In [19]:
def Evaluate_tweet(state:Tweetstate):
    prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template("You are a helpful assistant that evaluates tweets based on their quality and relevance to the given topic."),
        HumanMessagePromptTemplate.from_template("Evaluate the following tweet about the topic '{topic}': {tweet}. Provide feedback and indicate if it is 'approved' or 'needs_improvement'."),
    ])
    response = structured_evaluator_llm.invoke(prompt)
    
    return {"evaluation": response.evaluation, "feedback": response.feedback}

In [20]:
def Optimize_tweet(state:Tweetstate):
    prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template("You are a helpful assistant that optimizes tweets based on feedback provided."),
        HumanMessagePromptTemplate.from_template(f"Optimize the following tweet based on the feedback provided. Topic: {state['topic']}, Tweet: {state['tweet']}, Feedback: {state['feedback']}"),
    ])
    response = optimizer_llm.invoke(prompt).content
    iteration = state['iteration'] + 1
    return {"tweet": response, "iteration": iteration}

In [21]:
def route_evaluation(state:Tweetstate):
    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iterations']:
        return 'approved'
    else:
        return 'needs_improvement'

In [23]:
graph = StateGraph(Tweetstate)

graph.add_node('Generate', Generate_tweet)
graph.add_node('Evaluate', Evaluate_tweet)
graph.add_node('Optimize', Optimize_tweet)

graph.add_edge(START, 'Generate')
graph.add_edge('Generate', 'Evaluate')
graph.add_conditional_edges('Evaluate', route_evaluation, {'approved': END, 'needs_improvement': 'Optimize'})
graph.add_edge('Optimize', 'Evaluate')

workflow = graph.compile()


/Users/sarthakgulati/Documents/AIML-Track/03_langGraph/.venv/lib/python3.13/site-packages/langgraph/graph/state.py:116: UserWarning: Invalid state_schema: <function Tweetstate at 0x1147f9940>. Expected a type or Annotated[type, reducer]. Please provide a valid schema to ensure correct updates.
 See: https://langchain-ai.github.io/langgraph/reference/graphs/#stategraph
  warnings.warn(


In [24]:
initial_state = {
    'topic': 'Artificial Intelligence',
    'tweet': '',
    'evaluation': '',
    'feedback': '',
    'iteration': 0,
    'max_iterations': 3
}

In [25]:
workflow.invoke(initial_state)

ValueError: Invalid input type <class 'langchain_core.prompts.chat.ChatPromptTemplate'>. Must be a PromptValue, str, or list of BaseMessages.